# W03 · 02 LJ 유체 64입자 NVE 1,000스텝 (numpy)
목표: 원자 64개 · 주기 상자 · 컷오프 8.5 Å · velocity-Verlet Δt 1 fs → 전체 에너지 E = K + U 가 보존되는지 · 온도가 요동하는지 확인
출력: `out/lj_energy.csv` (step, t_fs, E_kin, E_pot, E_tot, T_K) · `out/lj_fluid_final.npz`
소요: CPU 약 1~3분 (내 컴퓨터 · mdai)

실행(기본): week03 폴더에서 연 Claude Code 에 「`@notebooks/02_lj_fluid_nve.ipynb` 를 실행하고 결과를 요약해」 → 승인 → `python ../run_nb.py notebooks/02_lj_fluid_nve.ipynb` · 그림을 직접 보려면 `jupyter lab`

In [ ]:
# 기본 = 내 컴퓨터 mdai 환경 · Claude Code 가 week03 폴더에서 실행 · 아래 google.colab 분기는 Colab(선택)에서만 동작
import os, sys
if "google.colab" in sys.modules and not os.path.exists("scripts/lj_verlet.py"):
    !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git repo && %cd repo/week03
import numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "scripts")
from lj_verlet import *   # EPS SIG M CONV KB lj_u lj_f pair_step bond_sweep lj_fluid OUT
print("ε", EPS, "σ", SIG, "m", M)

## 1. 실행 (scripts/lj_verlet.py 의 lj_fluid 와 같은 함수)

In [ ]:
summary = lj_fluid(n_side=4, rho=0.012, T0=300.0, dt=1.0, steps=1000, rc=8.5, seed=1)
print(json.dumps(summary, indent=1)) if (json:=__import__("json")) else None
# expected_output/lj_fluid_summary.json: drift_rel −1e−5 · E_std_rel 1e−5 · T_mean 213.9

## 2. 에너지 · 온도 시계열

In [ ]:
d = np.loadtxt(OUT / "lj_energy.csv", delimiter=",", skiprows=1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(d[:,1], d[:,2], label="E_kin"); ax[0].plot(d[:,1], d[:,3], label="E_pot"); ax[0].plot(d[:,1], d[:,4], "k", label="E_tot")
ax[0].set_xlabel("t (fs)"); ax[0].set_ylabel("kcal/mol"); ax[0].legend()
ax[1].plot(d[:,1], d[:,5]); ax[1].set_xlabel("t (fs)"); ax[1].set_ylabel("T (K)")
fig.tight_layout(); fig.savefig(OUT / "lj_energy.png", dpi=120); plt.show()
drift = (d[-1,4]-d[0,4])/abs(d[0,4]); print(f"에너지 드리프트 {drift*100:.4f} %  (목표 < 0.1 %)")

## 3. 읽기
- E_tot 는 거의 수평(드리프트 < 0.1 %) · K 와 U 는 서로 반대로 요동 → 운동에너지 ⇄ 퍼텐셜 에너지 교환
- 온도는 순간값이 요동한다 → 물성은 시간 평균(슬라이드 13) · 초기 T0 300 K 가 평균 약 214 K 로 내려간 이유 = 격자 배치의 퍼텐셜로 에너지가 분배(등분배) — 평형화는 W05
- Δt 를 4 fs 로 바꿔 다시 실행 → 드리프트 비교 (LJ 유체는 결합이 없어 C–C 결합보다 훨씬 관대하다)

In [ ]:
# 선택: Δt 4 fs 비교
s4 = lj_fluid(dt=4.0, steps=250, seed=1); print("dt 4 fs:", s4["drift_rel"], s4["E_std_rel"])